# Transparency Index Calculation

This transparency index is based on the index suggested in Irvin, R. ([link to pdf in repo](How%20Dark%20Is%20It_%20An%20Investigation%20of%20Dark%20Money%20Operations%20in%20US%20Nonprofit%20Political%20Advocacy%20Organizations.pdf), or [DOI](https://doi.org/10.1515/npf-2022-0032)).

The index is comprised of 9 components, and its composite score ranges from 0-9 (each component is scaled to a range of 0-1 inclusive). However, they invert the range, such that a higher score is indicative of *less* transparency. As a result, a score of 9 within this context would indicate a sever lack of transparency, and a high likelihood that the organization is a dark money organization. The author stresses that a high score on any single component is not indicative of covert operations, and that determinations should to the final total index score.

It is also worth noting that the author's dataset is limited to one policy domain (economics), whereas our project does not make any prior domain restrictions. Similarly, we are calculating the transparency index (and its components) for each filing year present for every 501(C)(4). This will allow us to expand on Irvin, R.'s work, through incorporating historical data across a much broader range of 501(C)(4)s.

The greatest limitation that arises from our systematic approach is that we did not have the time to perform a complete copy of their index. Their third component - the verification of an organization through website analysis - requires the counting of the words on the website of the C4, or the related C3 or 527 organizations if the C4 did not have one. The IRS does not enforce any standardization or verification for the filer-provided websites in their returns. As such, we would have to develop a method to systematically clean and verify urls for hundreds of thousands of filings - as an organization's website url could change over time.

This limitation lead us to identify a weakness with this component: it is inherently non-historical. If an organization shuts down, changes its website domain, or changes its content, it is not possible to get an accurate historical score for this component. However, we acknowledge that the author never suggested that these components be used outside of a single point in time, and that the limitation is unique to our application.

To attempt to retain some level of website verification, we applied a "semantic verification" process for the supplied urls, meaning, as long as the supplied websites matched the general url structure well enough, then they would be considered "valid". We could not obtain the websites for the 527 organizations, but were able to apply this semantic verification to the C4 and any of its related C3s.

Our application of this index can be found in the [`transparency_index.py`](..\extract\transparency_index.py) and the [`semantic_url_verification.py`](..\extract\semantic_url_verification.py) files.

Below is an outline of the transparency index components. With the exception of item 3, further descriptions and rationales are provided by the author on pages 112-117 of their article.

1. The number of board members of the C4 or its related C3s (the greatest of the two) normalized by 25
2. Binary representation whether or not an org has volunteers
3. Website verification: if the C4 does not have a valid website, then check the related C3s
4. Relation to 527s
5. Relation to C3s
6. Political expenses scaled by total expenses
7. Total salaries scaled by total expenses
8. Unrestricted net assets scaled by total expenses
9. Fundraising expenses scaled by total revenue

In [ ]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
# ! Update this URL accordingly
DB_PATH = Path("E:\\irs990_full.db")

from extract.transparency_index import (
    calculate_index_components,
    get_transparency_index_data,
)

In [3]:
conn = sqlite3.connect(DB_PATH)
df = get_transparency_index_data(conn, "full_dataset.csv")
df.head()

,filing_id,ein,total_revenue,total_expenses,total_assets,voting_members_independent,total_volunteers,website,total_salaries,unrestricted_net_assets_eoy,fundraising_expenses,lobbying,num_527s,num_c3s,max_board_size,political_grants
0,47,910839740,62778.0,45861.0,211659.0,0.0,0.0,N/A,0.0,211041.0,0.0,0,0.0,0.0,0.0,0.0
1,71,240789812,231379.0,192557.0,165519.0,237.0,7.0,,115494.0,163541.0,0.0,0,0.0,0.0,0.0,0.0
2,88,020672951,142409.0,83700.0,663596.0,7.0,35.0,N/A,0.0,663596.0,0.0,0,0.0,0.0,0.0,0.0
3,119,391678012,1958278.0,2063134.0,576652.0,11.0,284.0,WWW.PPAWI.ORG,0.0,157498.0,0.0,0,2.0,1.0,25.0,293006.0
4,173,410948789,223236.0,205371.0,825449.0,9.0,0.0,,0.0,-184815.0,0.0,0,0.0,0.0,0.0,0.0


In [61]:
index_df = calculate_index_components(df, conn)
index_df

,filing_id,ein,board_members,volunteers,website,related_to_527s,related_to_C3s,total_salaries,unrestricted_net_assets,fundraising_expenses,index
0,47,910839740,0.00,1,1,0,1,1.000000,0.000000,1.0,5.000000
1,71,240789812,1.00,0,1,0,1,0.400209,0.716896,1.0,5.117105
2,88,020672951,0.28,0,1,0,1,1.000000,0.000000,1.0,4.280000
3,119,391678012,1.00,0,0,1,0,1.000000,0.974554,1.0,4.974554
4,173,410948789,0.36,1,1,0,1,1.000000,1.000000,1.0,6.360000
...,...,...,...,...,...,...,...,...,...,...,...
160330,4472459,680047325,0.24,0,0,0,1,1.000000,0.000000,1.0,3.240000
160331,4472493,822575169,0.12,1,0,0,1,1.000000,0.565137,1.0,4.685137
160332,4472498,042562763,0.36,1,0,0,1,0.559706,0.718905,1.0,4.638611
160333,4472507,460450254,0.28,1,0,0,1,0.216716,0.000000,1.0,3.496716


In [62]:
index_df.describe()

,filing_id,board_members,volunteers,website,related_to_527s,related_to_C3s,total_salaries,unrestricted_net_assets,fundraising_expenses,index
count,1.603350e+05,160335.000000,160335.000000,160335.000000,160335.000000,160335.000000,160335.000000,160335.000000,160335.000000,160335.000000
mean,2.205773e+06,0.181877,0.738223,0.568160,0.012125,0.932610,0.904770,0.485377,0.971359,4.794501
std,1.275787e+06,0.271331,0.439603,0.495334,0.109443,0.250697,0.215657,0.382986,0.164439,0.865857
min,4.700000e+01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.122532e+06,0.000000,0.000000,0.000000,0.000000,1.000000,0.984531,0.000000,1.000000,4.226314
50%,2.176626e+06,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.565408,1.000000,4.960556
75%,3.300071e+06,0.280000,1.000000,1.000000,0.000000,1.000000,1.000000,0.845585,1.000000,5.432969
max,4.472542e+06,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,7.427619
